[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prefect-certified/notebooks/day-06-results-artifacts.ipynb#scrollTo=b1c2d3e4)

---
# Day 6 · Results, Artifacts, and State Persistence
**certified-journeys / prefect-certified** · Practice · Prefect for Data Engineers

> **Goal for today:** By the end of this notebook you can configure `LocalFileSystemResultStorage` to persist task results between runs, use `persist_result=True` to force-write individual results to disk, and publish both table and link artifacts that appear in the Prefect UI alongside each run.

In [ ]:
%pip install -q "prefect>=2.14"

## Step 1 · What are results and why persist them?

A Prefect **result** is the serialised return value of a task or flow run. By default, results are held in memory for the duration of the run and discarded afterward.

**Persisting results** writes them to a storage backend so they can be:

| Use case | Benefit |
|---|---|
| Cache (`cache_key_fn`) | Prefect reads the result instead of re-running the task |
| Crash recovery | Re-attach to a partially-completed run and skip done tasks |
| Cross-run data sharing | A downstream run can load a result written by a previous run |
| Debugging | Inspect exactly what a task produced without re-running |

**Storage backends** (Prefect 2.x):

| Backend | Class | When to use |
|---|---|---|
| Local filesystem | `LocalFileSystemResultStorage` | Development / CI |
| S3 | `S3ResultStorage` | Production on AWS |
| GCS | `GCSResultStorage` | Production on GCP |
| Azure Blob | `AzureBlobResultStorage` | Production on Azure |

> **Results ≠ Artifacts.** Results are for pipeline wiring and caching. Artifacts are for human-readable reporting in the UI. Both are covered today.

In [ ]:
import pathlib
from prefect import flow, task
from prefect.filesystems import LocalFileSystem
from prefect.results import LocalFileSystemResultStorage


# ── Configure a local result storage backend ──────────────────────────────────

# All result files will land under /tmp/prefect-results/
RESULT_DIR = "/tmp/prefect-results"
pathlib.Path(RESULT_DIR).mkdir(parents=True, exist_ok=True)

result_storage = LocalFileSystemResultStorage(path=RESULT_DIR)
print(f"Result storage configured → {RESULT_DIR}")
print(f"Type: {type(result_storage).__name__}")

### What just happened?

- `LocalFileSystemResultStorage` points Prefect at a directory for writing result files.
- Each persisted result is written as a serialised JSON file under that path with a content-addressed filename.
- **In production** you'd swap this for `S3ResultStorage(bucket="my-bucket", key_prefix="prefect/results/")` — the task code is identical.
- `pathlib.Path.mkdir(parents=True, exist_ok=True)` ensures the directory exists before Prefect tries to write.

## Step 2 · `persist_result=True` — force-persist a single task result

`persist_result` can be set at three levels:

```python
# 1. Task decorator level (applies to every call of this task)
@task(persist_result=True, result_storage=result_storage)
def my_task(): ...

# 2. Flow decorator level (applies to the flow's own result)
@flow(persist_result=True, result_storage=result_storage)
def my_flow(): ...

# 3. Global default via PREFECT_RESULTS_PERSIST_BY_DEFAULT=true env var
```

When `persist_result=True` is set on a task:
1. After the task body returns, Prefect serialises the return value.
2. The serialised bytes are written to the storage backend.
3. The **result reference** (a path or URI) is stored in the run's state — not the value itself.
4. When another task or flow accesses the result, Prefect reads the file and deserialises it.

In [ ]:
import json
import pathlib
from prefect import flow, task
from prefect.results import LocalFileSystemResultStorage


RESULT_DIR = "/tmp/prefect-results"
pathlib.Path(RESULT_DIR).mkdir(parents=True, exist_ok=True)
result_storage = LocalFileSystemResultStorage(path=RESULT_DIR)


@task(
    persist_result=True,           # force write this task's result to disk
    result_storage=result_storage, # where to write it
)
def compute_aggregates(data: list[float]) -> dict:
    """Computes summary statistics — result persisted to disk."""
    total   = sum(data)
    count   = len(data)
    average = round(total / count, 4) if count else 0.0
    minimum = min(data) if data else 0.0
    maximum = max(data) if data else 0.0
    result  = {"total": total, "count": count, "avg": average, "min": minimum, "max": maximum}
    print(f"  compute_aggregates → {result}")
    return result


@flow(name="persist-demo", log_prints=True)
def persist_demo_flow():
    sample_data = [4.5, 7.2, 3.1, 9.8, 6.0, 2.4, 5.5]
    stats = compute_aggregates(sample_data)
    print(f"  Flow received stats: {stats}")
    return stats


persist_demo_flow()

# Verify files were written to disk
result_files = list(pathlib.Path(RESULT_DIR).glob("**/*"))
print(f"\nFiles in {RESULT_DIR}:")
for f in result_files:
    print(f"  {f} ({f.stat().st_size} bytes)" if f.is_file() else f"  {f}/")

### What just happened?

- `persist_result=True` caused Prefect to write the task's return value to `RESULT_DIR` as a serialised file.
- The file listing confirms at least one result file was created — Prefect uses content-addressed filenames.
- The flow itself received the **deserialised value** as if nothing special happened — persistence is transparent to caller code.
- **Key insight:** `persist_result` is independent of caching — you can persist without caching and cache without persisting.

## Step 3 · Reading a persisted result in a later run

Once a result is on disk, a later run can load it by combining `persist_result=True` with `cache_key_fn`. This avoids re-running expensive computation when the inputs haven't changed.

The important distinction:

| Feature | Purpose | Storage |
|---|---|---|
| `persist_result=True` | Write the result so it survives the run | Result backend (local / S3 / GCS) |
| `cache_key_fn` | Use a stored result instead of re-running | Same result backend |
| Together | Persist once, reuse many times across runs | Result backend |

In [ ]:
from datetime import timedelta
from prefect import flow, task
from prefect.tasks import task_input_hash
from prefect.results import LocalFileSystemResultStorage


result_storage = LocalFileSystemResultStorage(path="/tmp/prefect-results")

_call_count = 0


@task(
    persist_result=True,
    result_storage=result_storage,
    cache_key_fn=task_input_hash,       # reuse persisted result on same inputs
    cache_expiration=timedelta(hours=1),
)
def expensive_feature_engineering(dataset_id: str) -> list[float]:
    """Simulates a slow feature computation — result persisted + cached."""
    global _call_count
    _call_count += 1
    import time, math
    print(f"  [RUNNING] expensive_feature_engineering call #{_call_count} for {dataset_id!r}")
    time.sleep(0.5)  # simulate expensive work
    return [round(math.sin(i) * 100, 2) for i in range(1, 6)]


@flow(name="feature-cache-demo", log_prints=True)
def feature_cache_demo_flow(dataset_id: str):
    features = expensive_feature_engineering(dataset_id)
    print(f"  Features for {dataset_id!r}: {features}")
    return features


print("=== First run (cache miss — result written to disk) ===")
feature_cache_demo_flow(dataset_id="customers_q1")

print("\n=== Second run (cache hit — result loaded from disk) ===")
feature_cache_demo_flow(dataset_id="customers_q1")

print(f"\nTotal task body executions: {_call_count} (expected 1 — second run used cached result)")

### What just happened?

- The first run wrote the result to `/tmp/prefect-results/` and stored a cache entry keyed on `(task_name, dataset_id)`.
- The second run found the cache hit and **loaded the result from disk** — `_call_count` remained 1.
- **This pattern spans process restarts**: as long as the result file exists on disk, a fresh Python process can use the cache.
- In production with S3 storage, this means multiple workers or Colab sessions share the same result cache.

## Step 4 · Creating a table artifact with `create_table_artifact`

**Artifacts** are human-readable records published alongside a flow run — visible in the Prefect UI's Artifacts tab and accessible via the API.

Three built-in artifact creators:

| Function | Output | When to use |
|---|---|---|
| `create_table_artifact()` | Rendered HTML table | Summaries, row counts, stats |
| `create_link_artifact()` | Clickable hyperlink | External dashboards, S3 URLs |
| `create_markdown_artifact()` | Rendered Markdown | Free-form reports, checklists |

> Artifacts appear in the Prefect UI under **Artifacts** in the left nav and also on each individual run's detail page.

In [ ]:
from prefect import flow, task
from prefect.artifacts import create_table_artifact


@task
def compute_pipeline_stats(rows: list[dict]) -> dict:
    """Computes per-region summary stats from a list of order dicts."""
    from collections import defaultdict
    summary: dict[str, dict] = defaultdict(lambda: {"count": 0, "total_amount": 0.0})
    for row in rows:
        region = row.get("region", "unknown")
        summary[region]["count"]        += 1
        summary[region]["total_amount"] += row.get("amount", 0.0)
    # Round totals
    return {
        region: {"count": v["count"], "total_amount": round(v["total_amount"], 2)}
        for region, v in summary.items()
    }


@task
async def publish_stats_table(stats: dict, run_label: str) -> None:
    """Publishes a Prefect table artifact from the stats dict."""
    # create_table_artifact expects a list of dicts (one dict per row)
    table_rows = [
        {"Region": region, "Orders": data["count"], "Total Amount ($)": data["total_amount"]}
        for region, data in sorted(stats.items())
    ]
    await create_table_artifact(
        key="pipeline-stats",               # stable key — latest version shown in UI
        table=table_rows,
        description=f"Order summary for run: {run_label}",
    )
    print(f"  Published table artifact with {len(table_rows)} rows")


@flow(name="table-artifact-demo", log_prints=True)
async def table_artifact_demo_flow():
    sample_orders = [
        {"id": 1, "region": "north", "amount": 120.50},
        {"id": 2, "region": "south", "amount": 85.00},
        {"id": 3, "region": "north", "amount": 200.00},
        {"id": 4, "region": "east",  "amount": 55.75},
        {"id": 5, "region": "south", "amount": 310.20},
        {"id": 6, "region": "east",  "amount": 90.00},
    ]

    stats = compute_pipeline_stats(sample_orders)
    print(f"  Raw stats: {stats}")

    await publish_stats_table(stats, run_label="2024-Q1")
    return stats


import asyncio
asyncio.run(table_artifact_demo_flow())

### What just happened?

- `create_table_artifact()` accepted a `list[dict]` — each dict is one table row, keys become column headers.
- The `key="pipeline-stats"` parameter gives the artifact a **stable identifier**; each new run creates a new version, and the UI shows the latest by default.
- In the Prefect UI you'd see a rendered HTML table under the run's Artifacts tab and also globally under Artifacts in the nav.
- **Artifacts are not the same as results**: they're published for human consumption, not for task-to-task data wiring.

## Step 5 · Creating a link artifact with `create_link_artifact`

`create_link_artifact` publishes a clickable URL in the Prefect UI. Common uses:

- Link to the S3 prefix where output files were written
- Link to a Grafana dashboard for the current run's metrics
- Link to a dbt Cloud run page
- Link to the source data file that was processed

```python
await create_link_artifact(
    key="output-s3",
    link="s3://my-bucket/runs/2024-01-15/output.parquet",
    description="Processed output file on S3",
)
```

In [ ]:
import pathlib
import json
from prefect import flow, task
from prefect.artifacts import create_link_artifact


@task
def write_output_file(data: list[dict], output_path: str) -> str:
    """Writes data to a JSON file and returns the absolute path."""
    path = pathlib.Path(output_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2))
    print(f"  Wrote {len(data)} records to {path}")
    return str(path.resolve())


@task
async def publish_output_link(file_path: str, description: str) -> None:
    """Publishes a link artifact pointing at the output file."""
    # In production: file_path would be an s3:// or https:// URL.
    # For local demo we use a file:// URI so the link is still clickable.
    link_url = f"file://{file_path}"
    await create_link_artifact(
        key="pipeline-output",
        link=link_url,
        description=description,
    )
    print(f"  Published link artifact → {link_url}")


@flow(name="link-artifact-demo", log_prints=True)
async def link_artifact_demo_flow(run_id: str = "run-001"):
    processed_records = [
        {"id": 1, "region": "north", "amount": 120.50, "processed": True},
        {"id": 2, "region": "south", "amount": 85.00,  "processed": True},
    ]
    output_path = f"/tmp/prefect-output/{run_id}/orders.json"

    written_path = write_output_file(processed_records, output_path)
    await publish_output_link(
        file_path=written_path,
        description=f"Processed orders for {run_id} — {len(processed_records)} records",
    )
    print(f"  Flow complete. Output at: {written_path}")


import asyncio
asyncio.run(link_artifact_demo_flow(run_id="run-2024-01"))

### What just happened?

- `create_link_artifact` accepted a `link` URL and a human-readable `description`.
- The artifact is attached to the current flow run — clicking it in the UI opens the linked URL.
- **Production pattern**: use an S3 pre-signed URL or GCS signed URL so the artifact link always works even outside the VPC.
- `key="pipeline-output"` means every run updates the *same* artifact key — the UI shows the latest alongside all historical versions.

## Step 6 · `create_markdown_artifact` — free-form run summaries

`create_markdown_artifact` renders arbitrary Markdown in the Prefect UI. It's the most flexible artifact type — use it for:

- **Run reports**: what was processed, how many rows, any anomalies
- **Data quality summaries**: pass/fail checks with counts
- **Deployment checklists**: which steps ran, which were skipped

The Markdown is rendered with full support for tables, code blocks, bold/italic, and lists.

In [ ]:
from prefect import flow, task
from prefect.artifacts import create_markdown_artifact


@task
def run_quality_checks(rows: list[dict]) -> dict:
    """Runs simple data quality checks; returns a results dict."""
    total       = len(rows)
    missing_amt = sum(1 for r in rows if r.get("amount") is None)
    negative    = sum(1 for r in rows if (r.get("amount") or 0) < 0)
    duplicates  = total - len({r["id"] for r in rows})
    return {
        "total": total,
        "missing_amount": missing_amt,
        "negative_amount": negative,
        "duplicate_ids": duplicates,
        "passed": missing_amt == 0 and negative == 0 and duplicates == 0,
    }


@task
async def publish_quality_report(checks: dict, pipeline_name: str, run_label: str) -> None:
    """Publishes a Markdown data-quality report as a Prefect artifact."""
    status_icon = "✅" if checks["passed"] else "⚠️"
    md = f"""## {status_icon} Data Quality Report — {pipeline_name}

**Run:** `{run_label}`  
**Status:** {'PASSED' if checks['passed'] else 'FAILED'}

### Check Results

| Check | Count | Status |
|---|---|---|
| Total rows | {checks['total']} | ℹ |
| Missing `amount` | {checks['missing_amount']} | {'✅' if checks['missing_amount'] == 0 else '❌'} |
| Negative `amount` | {checks['negative_amount']} | {'✅' if checks['negative_amount'] == 0 else '❌'} |
| Duplicate IDs | {checks['duplicate_ids']} | {'✅' if checks['duplicate_ids'] == 0 else '❌'} |

### Action Required
{'_No issues found. Pipeline output is reliable._' if checks['passed']
 else '- Investigate rows with missing or negative amounts before loading to the warehouse.'}
"""
    await create_markdown_artifact(
        key="data-quality-report",
        markdown=md,
        description=f"DQ report for {pipeline_name} ({run_label})",
    )
    print(f"  Published Markdown quality report (passed={checks['passed']})")


@flow(name="markdown-artifact-demo", log_prints=True)
async def markdown_artifact_demo_flow():
    clean_rows = [
        {"id": 1, "amount": 55.0},
        {"id": 2, "amount": 120.0},
        {"id": 3, "amount": 88.5},
    ]
    dirty_rows = [
        {"id": 1, "amount": 55.0},
        {"id": 1, "amount": 55.0},   # duplicate
        {"id": 3, "amount": -9.0},   # negative
    ]

    print("--- Clean data run ---")
    checks_clean = run_quality_checks(clean_rows)
    await publish_quality_report(checks_clean, "orders-pipeline", "2024-Q1-clean")

    print("--- Dirty data run ---")
    checks_dirty = run_quality_checks(dirty_rows)
    await publish_quality_report(checks_dirty, "orders-pipeline", "2024-Q1-dirty")


import asyncio
asyncio.run(markdown_artifact_demo_flow())

### What just happened?

- `create_markdown_artifact` accepted a raw Markdown string including a table, emoji, and conditional text.
- In the Prefect UI, this renders as a **formatted HTML page** attached to each run — great for on-call engineers who need a quick summary without reading logs.
- **Using f-strings to build the Markdown** is idiomatic — the logic (pass/fail) lives in Python, the presentation lives in the Markdown template.
- The artifact key `"data-quality-report"` accumulates versions — you can view the quality trend across runs in the UI.

## Step 7 · Results vs Artifacts — when to use each

This is the most important conceptual distinction of the day:

| Dimension | Results | Artifacts |
|---|---|---|
| **Audience** | The pipeline itself (tasks reading other tasks' outputs) | Humans (engineers, stakeholders, on-call) |
| **Format** | Serialised Python object (pickle / JSON) | Human-readable (table, link, Markdown) |
| **Storage** | Result backend (local FS, S3, GCS) | Prefect database / UI |
| **Use case** | Caching, crash recovery, cross-run data sharing | Run reports, data quality, audit trail |
| **Visibility** | Not visible in UI (only as metadata) | Shown prominently on run detail page |
| **API** | `persist_result`, `result_storage`, `cache_key_fn` | `create_table_artifact`, `create_link_artifact`, `create_markdown_artifact` |

> **Rule of thumb:** If a downstream task needs it → Result. If a human needs it → Artifact. You often want both.

In [ ]:
import pathlib
import json
from datetime import timedelta
from prefect import flow, task
from prefect.tasks import task_input_hash
from prefect.results import LocalFileSystemResultStorage
from prefect.artifacts import create_table_artifact, create_markdown_artifact


result_storage = LocalFileSystemResultStorage(path="/tmp/prefect-results")
pathlib.Path("/tmp/prefect-results").mkdir(parents=True, exist_ok=True)


@task(
    persist_result=True,              # result: for downstream task wiring
    result_storage=result_storage,
    cache_key_fn=task_input_hash,
    cache_expiration=timedelta(hours=6),
)
def score_customers(customers: list[dict]) -> list[dict]:
    """Scores customers — expensive, result persisted + cached."""
    import math
    scored = [
        {**c, "score": round(math.log1p(c.get("spend", 1)) * 10, 2)}
        for c in customers
    ]
    print(f"  Scored {len(scored)} customers")
    return scored


@task
async def publish_scoring_report(scored: list[dict], run_label: str) -> None:
    """Artifact: human-readable summary of the scoring run."""
    total   = len(scored)
    avg_scr = round(sum(c["score"] for c in scored) / total, 2) if total else 0
    top3    = sorted(scored, key=lambda c: c["score"], reverse=True)[:3]

    # Table artifact — machine-readable rows
    await create_table_artifact(
        key="customer-scores",
        table=[{"Customer": c["name"], "Spend": c["spend"], "Score": c["score"]} for c in top3],
        description=f"Top 3 customers by score — {run_label}",
    )

    # Markdown artifact — prose summary
    md = f"""## Customer Scoring Summary — {run_label}

- **Customers scored:** {total}
- **Average score:** {avg_scr}
- **Top scorer:** {top3[0]['name']} ({top3[0]['score']})

> Scores are log-scaled from lifetime spend. Re-run weekly or when spend data is refreshed.
"""
    await create_markdown_artifact(
        key="scoring-summary",
        markdown=md,
        description=f"Scoring run summary — {run_label}",
    )
    print(f"  Published table + markdown artifacts for {run_label}")


@flow(name="full-scoring-pipeline", log_prints=True)
async def full_scoring_pipeline(run_label: str = "2024-W01"):
    customers = [
        {"id": 1, "name": "Alice",   "spend": 1500},
        {"id": 2, "name": "Bob",     "spend": 300},
        {"id": 3, "name": "Carol",   "spend": 2200},
        {"id": 4, "name": "Dave",    "spend": 850},
        {"id": 5, "name": "Eve",     "spend": 4100},
    ]
    # Result: persisted for caching and downstream wiring
    scored = score_customers(customers)

    # Artifact: human-readable report in the UI
    await publish_scoring_report(scored, run_label)

    return scored


import asyncio
asyncio.run(full_scoring_pipeline())

### What just happened?

- `score_customers` uses **both** `persist_result` (for caching) and returns data the flow uses — the result is for pipeline wiring.
- `publish_scoring_report` creates **both** a table artifact (structured data) and a Markdown artifact (prose summary) — both are for humans.
- **The two systems are additive**: a single task can produce a result (for machines) and trigger artifact creation (for humans) in the same run.
- The `key` on each artifact accumulates versions — the UI shows a timeline of every scoring run.

In [ ]:
# ─── Challenge ────────────────────────────────────────────────────────────────
# Challenge: Build a complete "sensor data pipeline" that demonstrates all of
# today's concepts together.
#
# Requirements:
#
#   1. ingest_sensor_data(sensor_id: str) -> list[dict]
#        Return a list of 10 simulated sensor readings:
#        [{"ts": i, "value": random float between 0-100}]
#        MUST be decorated with persist_result=True, cache_key_fn=task_input_hash,
#        cache_expiration=timedelta(minutes=10).
#
#   2. detect_anomalies(readings: list[dict], threshold: float) -> dict
#        Return {"anomalies": [readings above threshold], "pct_anomalous": float}.
#
#   3. publish_sensor_report(sensor_id, readings, anomaly_result) -> None (async)
#        Create:
#          a) A table artifact (key="sensor-readings") showing the raw readings.
#          b) A markdown artifact (key="sensor-anomaly-report") summarising
#             the anomaly count, percentage, and whether action is needed
#             (action needed if pct_anomalous > 20%).
#
#   4. A parent async flow that calls all three, plus a link artifact
#      (key="sensor-dashboard") pointing to
#      "https://grafana.example.com/sensor/{sensor_id}".
#
# Test by running with sensor_id="temp-01" and threshold=70.0.
# ──────────────────────────────────────────────────────────────────────────────

import asyncio
import random
from datetime import timedelta
from prefect import flow, task
from prefect.tasks import task_input_hash
from prefect.results import LocalFileSystemResultStorage
from prefect.artifacts import create_table_artifact, create_markdown_artifact, create_link_artifact
import pathlib

result_storage = LocalFileSystemResultStorage(path="/tmp/prefect-results")
pathlib.Path("/tmp/prefect-results").mkdir(parents=True, exist_ok=True)


# TODO: implement ingest_sensor_data (persist_result=True + cache)


# TODO: implement detect_anomalies


# TODO: implement publish_sensor_report (table + markdown artifacts)


# TODO: implement parent async flow with link artifact


# Uncomment to test:
# asyncio.run(sensor_pipeline(sensor_id="temp-01", threshold=70.0))

---
## Day 6 key concepts recap

| Concept | What to remember |
|---|---|
| `LocalFileSystemResultStorage` | Points Prefect at a local directory for writing serialised results |
| `persist_result=True` | Force-writes a task's return value to the result backend after each run |
| `persist_result` + `cache_key_fn` | Persist once, reuse across runs — cache check reads from disk |
| `create_table_artifact` | Publishes a rendered HTML table in the Prefect UI |
| `create_link_artifact` | Publishes a clickable URL (S3, dashboards, external reports) |
| `create_markdown_artifact` | Publishes free-form rendered Markdown — best for prose run summaries |
| Results vs Artifacts | Results = machine-to-machine; Artifacts = machine-to-human |

> **Tip:** Artifacts are surfaced directly in the Prefect UI alongside each run — use `create_markdown_artifact()` to publish a human-readable summary of what your pipeline produced.

---
## What's next
**Day 7** → Deployments Basics — package your flows as deployments with schedules, parameters, and work pools so they run without manual intervention.

Mark Day 6 complete in your [tracker](../index.html).